# Shapesphysical Hidden-State Results

This notebook is for interpreting the hidden-state structure-comparison runs on `shapesphysical`.

It focuses on:
- same-TR versus lagged LM hidden-state variants
- summary metric heatmaps
- TR-by-TR representational similarity heatmaps for brain and LM predictions
- top parcels and shared predictors

The notebook auto-detects whichever hidden-state output folders are available locally and skips missing runs.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "structure_comparison").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing structure_comparison/")


ROOT = find_repo_root()
OUTPUT_ROOT = ROOT / "structure_comparison" / "outputs"

RUN_CANDIDATES = [
    {
        "key": "gemma_2_2b_same",
        "label": "Gemma 2 2B Hidden Same-TR",
        "path": OUTPUT_ROOT / "final_output_retry_gemma_2_2b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state",
        "variant": "same_tr",
    },
    {
        "key": "gemma_2_2b_lagged",
        "label": "Gemma 2 2B Hidden Lagged-LM",
        "path": OUTPUT_ROOT / "final_output_retry_gemma_2_2b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state_lagged_lm",
        "variant": "lagged_lm",
    },
    {
        "key": "gemma_2_9b_same",
        "label": "Gemma 2 9B Hidden Same-TR",
        "path": OUTPUT_ROOT / "final_output_retry_gemma_2_9b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state",
        "variant": "same_tr",
    },
    {
        "key": "gemma_2_9b_lagged",
        "label": "Gemma 2 9B Hidden Lagged-LM",
        "path": OUTPUT_ROOT / "final_output_retry_gemma_2_9b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state_lagged_lm",
        "variant": "lagged_lm",
    },
    {
        "key": "llama_3_1_8b_same",
        "label": "Llama 3.1 8B Hidden Same-TR",
        "path": OUTPUT_ROOT / "final_output_retry_llama_3_1_8b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state",
        "variant": "same_tr",
    },
    {
        "key": "llama_3_1_8b_lagged",
        "label": "Llama 3.1 8B Hidden Lagged-LM",
        "path": OUTPUT_ROOT / "final_output_retry_llama_3_1_8b_shapesphysical_cleaned_batches00_05_average_all_layers_final_hidden_state_lagged_lm",
        "variant": "lagged_lm",
    },
]


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


available_runs = []
missing_runs = []
for spec in RUN_CANDIDATES:
    summary_path = spec["path"] / "all_layers" / "summary.json"
    analysis_path = spec["path"] / "analysis_summary.json"
    if summary_path.exists():
        available_runs.append(
            {
                **spec,
                "summary_path": summary_path,
                "analysis_path": analysis_path if analysis_path.exists() else None,
                "summary": load_json(summary_path),
            }
        )
    else:
        missing_runs.append(spec)


if not available_runs:
    raise RuntimeError("No hidden-state result folders are available locally.")


summary_rows = []
for run in available_runs:
    summary = run["summary"]
    summary_rows.append(
        {
            "label": run["label"],
            "variant": run["variant"],
            "predictor_count": summary["predictor_count"],
            "lm_design_predictor_count": summary.get("lm_design_predictor_count", np.nan),
            "lm_target_count": summary["lm_target_count"],
            "brain_r": summary["brain_mean_test_correlation"],
            "brain_r2": summary["brain_mean_test_r2"],
            "lm_r": summary["lm_mean_test_correlation"],
            "lm_r2": summary["lm_mean_test_r2"],
            "rsa": summary["sample_rsa_correlation"],
            "feature_corr": summary["feature_importance_correlation"],
            "path": str(run["path"]),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["label"]).reset_index(drop=True)


In [ ]:
display(Markdown("## Available Hidden-State Runs"))
display(summary_df)

if missing_runs:
    display(Markdown("## Missing Hidden-State Runs"))
    display(
        pd.DataFrame(
            [{"label": run["label"], "expected_path": str(run["path"])} for run in missing_runs]
        )
    )


In [ ]:
METRIC_COLUMNS = ["brain_r", "brain_r2", "lm_r", "lm_r2", "rsa", "feature_corr"]


def plot_metric_heatmap(df: pd.DataFrame, metric_columns: list[str], title: str) -> None:
    values = df[metric_columns].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(1.6 * len(metric_columns) + 2, 0.55 * len(df) + 1.8))
    im = ax.imshow(values, aspect="auto", cmap="YlGnBu")
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xticks(np.arange(len(metric_columns)))
    ax.set_xticklabels(metric_columns, rotation=30, ha="right")
    ax.set_yticks(np.arange(len(df)))
    ax.set_yticklabels(df["label"])

    for row_idx in range(values.shape[0]):
        for col_idx in range(values.shape[1]):
            value = values[row_idx, col_idx]
            ax.text(col_idx, row_idx, f"{value:.3f}", ha="center", va="center", fontsize=9)

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


plot_metric_heatmap(summary_df, METRIC_COLUMNS, "Hidden-State Summary Metrics")


In [ ]:
comparison_cols = ["label", "brain_r", "brain_r2", "lm_r", "lm_r2", "rsa", "feature_corr"]
same_df = summary_df[summary_df["variant"] == "same_tr"].copy()
lagged_df = summary_df[summary_df["variant"] == "lagged_lm"].copy()

display(Markdown("## Same-TR Runs"))
display(same_df[comparison_cols])

display(Markdown("## Lagged-LM Runs"))
display(lagged_df[comparison_cols])


## What Correlation Versus RSA Means

These metrics are not measuring the same thing.

- `brain_r` and `lm_r` ask whether the model predicts the correct target dimensions well on average.
- `rsa` asks whether the **pattern of similarity across TRs** looks the same in the brain and LM prediction spaces.

So a run can have:
- strong per-target prediction correlation
- but weaker RSA

if it gets many target dimensions roughly right while still organizing the TR-by-TR geometry differently.

The next sections are meant to show exactly **where** the runs look similar and where they diverge.


In [ ]:
pairwise_delta_rows = []
for left_idx in range(len(summary_df)):
    for right_idx in range(left_idx + 1, len(summary_df)):
        left = summary_df.iloc[left_idx]
        right = summary_df.iloc[right_idx]
        pairwise_delta_rows.append(
            {
                "left": left["label"],
                "right": right["label"],
                "delta_brain_r": right["brain_r"] - left["brain_r"],
                "delta_brain_r2": right["brain_r2"] - left["brain_r2"],
                "delta_lm_r": right["lm_r"] - left["lm_r"],
                "delta_lm_r2": right["lm_r2"] - left["lm_r2"],
                "delta_rsa": right["rsa"] - left["rsa"],
                "delta_feature_corr": right["feature_corr"] - left["feature_corr"],
            }
        )

pairwise_delta_df = pd.DataFrame(pairwise_delta_rows)
display(Markdown("## Pairwise Metric Differences"))
display(pairwise_delta_df)


In [ ]:
def average_rows_by_tr(values: np.ndarray, tr_indices: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    tr_indices = np.asarray(tr_indices, dtype=int)
    unique_trs = np.unique(tr_indices)
    averaged = []
    for tr in unique_trs:
        averaged.append(values[tr_indices == tr].mean(axis=0))
    return np.asarray(averaged, dtype=np.float32), unique_trs


def compute_similarity_matrix(values: np.ndarray) -> np.ndarray:
    if values.shape[0] < 2:
        raise ValueError("Need at least two samples to compute a similarity matrix.")
    matrix = np.corrcoef(values)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)
    return matrix


def load_prediction_matrices(run_path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray] | None:
    brain_path = run_path / "all_layers" / "brain_final_model.npz"
    lm_path = run_path / "all_layers" / "lm_final_model.npz"
    if not brain_path.exists() or not lm_path.exists():
        return None

    brain_npz = np.load(brain_path, allow_pickle=False)
    lm_npz = np.load(lm_path, allow_pickle=False)

    brain_avg, brain_trs = average_rows_by_tr(brain_npz["predictions"], brain_npz["tr_indices"])
    lm_avg, lm_trs = average_rows_by_tr(lm_npz["predictions"], lm_npz["tr_indices"])
    shared_trs = np.intersect1d(brain_trs, lm_trs)
    if shared_trs.size == 0:
        return None

    brain_lookup = {int(tr): row for tr, row in zip(brain_trs, brain_avg)}
    lm_lookup = {int(tr): row for tr, row in zip(lm_trs, lm_avg)}
    brain_aligned = np.asarray([brain_lookup[int(tr)] for tr in shared_trs], dtype=np.float32)
    lm_aligned = np.asarray([lm_lookup[int(tr)] for tr in shared_trs], dtype=np.float32)
    return brain_aligned, lm_aligned, shared_trs


def plot_rsa_heatmaps(run_label: str, brain_values: np.ndarray, lm_values: np.ndarray, tr_indices: np.ndarray, max_trs: int = 80) -> None:
    if brain_values.shape[0] > max_trs:
        keep = np.arange(max_trs)
        brain_values = brain_values[keep]
        lm_values = lm_values[keep]
        tr_indices = tr_indices[keep]

    brain_sim = compute_similarity_matrix(brain_values)
    lm_sim = compute_similarity_matrix(lm_values)
    diff = lm_sim - brain_sim

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    items = [
        (brain_sim, f"{run_label}\nBrain Predicted Similarity", "YlGnBu", -1.0, 1.0),
        (lm_sim, f"{run_label}\nLM Predicted Similarity", "YlGnBu", -1.0, 1.0),
        (diff, f"{run_label}\nLM - Brain Similarity", "coolwarm", -1.0, 1.0),
    ]
    for ax, (matrix, title, cmap, vmin, vmax) in zip(axes, items):
        im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("TR index")
        ax.set_ylabel("TR index")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


rsa_ready_runs = []
for run in available_runs:
    matrices = load_prediction_matrices(run["path"])
    if matrices is not None:
        rsa_ready_runs.append((run, matrices))

display(Markdown(f"## Runs With Local Final-Model Artifacts For RSA Heatmaps: `{len(rsa_ready_runs)}`"))
display(pd.DataFrame([{"label": run["label"], "path": str(run["path"])} for run, _ in rsa_ready_runs]))


In [ ]:
for run, (brain_values, lm_values, shared_trs) in rsa_ready_runs:
    display(Markdown(f"## RSA Heatmaps: {run['label']}"))
    plot_rsa_heatmaps(run["label"], brain_values, lm_values, shared_trs, max_trs=80)


In [ ]:
def pairwise_run_comparisons(rsa_runs: list[tuple[dict, tuple[np.ndarray, np.ndarray, np.ndarray]]]) -> None:
    if len(rsa_runs) < 2:
        display(Markdown("## Pairwise Geometry Differences"))
        display(Markdown("Need at least two runs with local final-model artifacts for pairwise geometry comparison."))
        return

    display(Markdown("## Pairwise Geometry Differences"))
    for left_idx in range(len(rsa_runs)):
        for right_idx in range(left_idx + 1, len(rsa_runs)):
            left_run, (left_brain, left_lm, left_trs) = rsa_runs[left_idx]
            right_run, (right_brain, right_lm, right_trs) = rsa_runs[right_idx]
            shared = np.intersect1d(left_trs, right_trs)
            if shared.size < 3:
                continue

            left_brain_lookup = {int(tr): row for tr, row in zip(left_trs, left_brain)}
            left_lm_lookup = {int(tr): row for tr, row in zip(left_trs, left_lm)}
            right_brain_lookup = {int(tr): row for tr, row in zip(right_trs, right_brain)}
            right_lm_lookup = {int(tr): row for tr, row in zip(right_trs, right_lm)}

            left_brain_aligned = np.asarray([left_brain_lookup[int(tr)] for tr in shared], dtype=np.float32)
            left_lm_aligned = np.asarray([left_lm_lookup[int(tr)] for tr in shared], dtype=np.float32)
            right_brain_aligned = np.asarray([right_brain_lookup[int(tr)] for tr in shared], dtype=np.float32)
            right_lm_aligned = np.asarray([right_lm_lookup[int(tr)] for tr in shared], dtype=np.float32)

            max_trs = 80
            if shared.size > max_trs:
                keep = np.arange(max_trs)
                shared = shared[keep]
                left_brain_aligned = left_brain_aligned[keep]
                left_lm_aligned = left_lm_aligned[keep]
                right_brain_aligned = right_brain_aligned[keep]
                right_lm_aligned = right_lm_aligned[keep]

            left_brain_sim = compute_similarity_matrix(left_brain_aligned)
            right_brain_sim = compute_similarity_matrix(right_brain_aligned)
            left_lm_sim = compute_similarity_matrix(left_lm_aligned)
            right_lm_sim = compute_similarity_matrix(right_lm_aligned)

            fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
            brain_delta = right_brain_sim - left_brain_sim
            lm_delta = right_lm_sim - left_lm_sim
            items = [
                (brain_delta, f"Brain geometry delta\n{right_run['label']} minus {left_run['label']}"),
                (lm_delta, f"LM geometry delta\n{right_run['label']} minus {left_run['label']}"),
            ]
            for ax, (matrix, title) in zip(axes, items):
                im = ax.imshow(matrix, cmap="coolwarm", vmin=-1.0, vmax=1.0, aspect="auto")
                ax.set_title(title, fontsize=10)
                ax.set_xlabel("TR index")
                ax.set_ylabel("TR index")
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            plt.tight_layout()
            plt.show()


pairwise_run_comparisons(rsa_ready_runs)


In [ ]:
top_parcel_rows = []
for run in available_runs:
    for row in run["summary"].get("top_parcels", [])[:5]:
        top_parcel_rows.append(
            {
                "label": run["label"],
                "target_name": row["target_name"],
                "correlation": row["correlation"],
                "r2": row["r2"],
            }
        )

display(Markdown("## Top Brain Parcels (Top 5 per run)"))
display(pd.DataFrame(top_parcel_rows))

top_feature_rows = []
for run in available_runs:
    for row in run["summary"].get("top_shared_predictors", [])[:8]:
        top_feature_rows.append(
            {
                "label": run["label"],
                "feature_name": row["feature_name"],
                "brain_importance": row["brain_importance"],
                "lm_importance": row["lm_importance"],
            }
        )

display(Markdown("## Top Shared Predictors (Top 8 per run)"))
display(pd.DataFrame(top_feature_rows))


In [ ]:
display(Markdown("## Quick Read"))

best_lm = summary_df.sort_values("lm_r", ascending=False).iloc[0]
best_rsa = summary_df.sort_values("rsa", ascending=False).iloc[0]
best_feature = summary_df.sort_values("feature_corr", ascending=False).iloc[0]

display(
    Markdown(
        "\n".join(
            [
                f"- strongest LM correlation currently local: `{best_lm['label']}` with `lm_r = {best_lm['lm_r']:.3f}`",
                f"- strongest scalar RSA currently local: `{best_rsa['label']}` with `rsa = {best_rsa['rsa']:.3f}`",
                f"- strongest feature-overlap currently local: `{best_feature['label']}` with `feature_corr = {best_feature['feature_corr']:.3f}`",
                "- use the heatmaps above to judge whether the LM and brain predictions share similar TR-by-TR geometry or only similar global summary scores.",
            ]
        )
    )
)
